# upload zip folder of .lif files to Colab
a notebook to learn how we can use zip folders from our local computer

If you want your data to be directly compatoble with this notebook, generate a folder with two subfolders (training and data). Place all .lif files (downloaded from Omero) into the data folder. In the training folder generate four subfolders: DAAO, vGAT, Cre, Gphn. Into each of these folders place three subfolders: images, masks, and models.

Next compress the entire folder into one .zip file. 

In Colab, we can trigger an Upload Dialog: 

In [ ]:
from google.colab import files
uploaded = files.upload()  # triggers a file picker dialog in the browser

In [ ]:
import zipfile
from pathlib import Path
from PIL import Image
import io

ZIPFILE = 'images.zip'  # name of the uploaded zip
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}

def analyze_image(img, filename):
    """
    img is already an open PIL Image object.
    Add your measurements here.
    """
    width, height = img.size
    mode = img.mode

    measurements = {
        'filename': filename,
        'width_px': width,
        'height_px': height,
        'mode': mode,
        # add your own measurements here
    }
    return measurements

def walk_and_process_zip(zip_path):
    results = []

    with zipfile.ZipFile(zip_path, 'r') as z:
        # filter to image files only, skip macOS metadata files
        image_files = sorted([
            name for name in z.namelist()
            if Path(name).suffix.lower() in EXTENSIONS
            and not Path(name).name.startswith('._')  
        ])

        print(f"Found {len(image_files)} images in zip")

        for i, name in enumerate(image_files, 1):
            print(f"[{i}/{len(image_files)}] Processing: {Path(name).name}")
            try:
                with z.open(name) as f:
                    img_bytes = io.BytesIO(f.read())
                    with Image.open(img_bytes) as img:
                        img.load()  # force load before BytesIO is released
                        result = analyze_image(img, Path(name).name)
                        results.append(result)
                # image is freed from memory here
            except Exception as e:
                print(f"  ⚠️  Skipped {Path(name).name}: {e}")

    return results

# ── Run ───────────────────────────────────────────────────
results = walk_and_process_zip(ZIPFILE)
print(f"\nDone! Processed {len(results)} images.")

In [ ]:
import pandas as pd
from google.colab import files

df = pd.DataFrame(results)
df.to_csv('results.csv', index=False)
print(df.head())

files.download('results.csv')

in Colab, we need to connect to a GPU

on the to right, go to change runtime type and select T4 GPU

In [ ]:
pip install scikit-image

In [ ]:
pip install matplotlib

In [ ]:
!pip install apoc --no-deps
!pip install "scikit-learn" "pyclesperanto-prototype" "pandas" "numpy==2.4.4"

In [ ]:
from matplotlib import pyplot as plt
import apoc
from skimage.io import imread, imsave
import numpy as np

In [ ]:
import os
# Clone the repo if not already in Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/NBCimageAnalysis'):
        !git clone https://github.com/FilLieb/NBCimageAnalysis.git
    os.chdir('/content/NBCimageAnalysis/learning/')
    print(os.listdir('.'))

We have 4 channels and assign their names to a list:

In [ ]:
channels = ["DAAO", "Cre", "vGAT", "Gphn"]  # change index to switch between channels

let's check that is all correct...

In [ ]:
for channel in channels: 
    image_folder = '../training_4_channels/' + channel + '/images/'
    masks_folder = '../training_4_channels/' + channel + '/masks/'

    image_path = os.path.join(image_folder, os.listdir(image_folder)[0])
    image = imread(image_path)

    masks_path = os.path.join(masks_folder, os.listdir(masks_folder)[0])
    masks = imread(masks_path)


    f, a = plt.subplots(1,3, figsize=(15,5))
    a[0].imshow(image, cmap='gray')
    a[0].set_title("Image" + channel)

    a[1].imshow(masks, vmin=0, vmax=2)
    a[1].set_title("Masks" + channel)

    a[2].imshow(image, cmap='gray')
    a[2].contour(masks, colors='r', linewidths=0.5)
    a[2].set_title("Overlay" + channel)

plt.show()

we can now loop through this list to train all models...

In [ ]:
for channel in channels: 
    image_folder = '../training_4_channels/' + channel + '/images/'
    masks_folder = '../training_4_channels/' + channel + '/masks/'

     # this is where the model will be saved
    cl_filename = '../training_4_channels/' + channel + '/models/' + channel + '_object_model.cl'
    
    apoc.erase_classifier(cl_filename) # delete it if it was existing before

    # setup classifier and where it should be saved
    segmenter = apoc.ObjectSegmenter(opencl_filename=cl_filename,
                                     max_depth=5,
                                     num_ensembles=1000)

    # setup feature set used for training
    features = apoc.PredefinedFeatureSet.small_dog_log.value + " " + \
               apoc.PredefinedFeatureSet.medium_dog_log.value + " " + \
               apoc.PredefinedFeatureSet.large_dog_log.value

    # train classifier on folders
    apoc.erase_classifier(cl_filename)
    apoc.train_classifier_from_image_folders(
        segmenter,
        features,
        image = image_folder,
        ground_truth = masks_folder)
    
    print("Training completed and model saved to " + cl_filename)



finally we can test how the model performed

In [ ]:
for channel in channels: 
    image_folder = '../training_4_channels/' + channel + '/images/'

    image_path = os.path.join(image_folder, os.listdir(image_folder)[0])
    image = imread(image_path)

    segmenter = apoc.ObjectSegmenter(opencl_filename=cl_filename)
    labels = segmenter.predict(image)

    f, a = plt.subplots(1,3, figsize=(15,5))
    a[0].imshow(image, cmap='gray')
    a[0].set_title("Image" + channel)

    a[1].imshow(labels, vmin=0, vmax=2)
    a[1].set_title("Labels" + channel)

    a[2].imshow(image, cmap='gray')
    a[2].contour(labels, colors='r', linewidths=0.5)
    a[2].set_title("Overlay" + channel)

plt.show()

Now we can put things together and run our models on multiple images...

In [ ]:
pip install readlif

In [ ]:
from readlif.reader import LifFile

In [ ]:
# set paths
image_folder = '../data/2026group2/'

In [ ]:
def execute(file, root):
    print(f"Found image: {file}")
    folder_name = os.path.splitext(os.path.basename(file))[0] # removes the .lif extension
    folder_path = os.path.join(root, folder_name)
    os.makedirs(folder_path, exist_ok=True)

    lif = LifFile(file)

    for img in lif.get_iter_image():
        print(img.name, img.dims)
        channels_array = np.stack([np.array(channel) for channel in img.get_iter_c(t=0, z=0)])
        make_masks("DAAO", channels_array[0], folder_path, img.name)
        make_masks("vGAT", channels_array[1], folder_path, img.name)
        make_masks("Cre", channels_array[2], folder_path, img.name)
        make_masks("Gphn", channels_array[3], folder_path, img.name)


In [ ]:
# make mask per channel
def make_masks(channel, array, path, image):
    print("Generating masks...")

    out_file = os.path.join(path, image + '_' + channel + '_labels.tif')
    
    cl_filename = '../training_4_channels/' + channel + '/models/' + channel + '_object_model.cl'
    segmenter = apoc.ObjectSegmenter(opencl_filename=cl_filename)
    labels = segmenter.predict(array)

    imsave(out_file, labels)

In [ ]:
for root, dirs, files in os.walk(image_folder):
    for file in files:
        if file.endswith("High Density.lif"):
            full_path = os.path.join(root, file)
            print(full_path)
            execute(full_path, root)

print("...completed.")


In [ ]:
for root, dirs, files in os.walk(image_folder):
    for file in files:
        if file.endswith("_labels.tif"):
            label_path = os.path.join(root, file)
            label = imread(label_path)
            folder_name = os.path.basename(root)
            plt.figure()
            plt.imshow(label)
            plt.title("predicted labels - " + folder_name + ": " + file)
            plt.show()
